# Qwen2.5-VL Original Split — Sequence-Level Pipeline with PMI

Original split logic:

1. score GT±WINDOW candidates
2. proxy_bias = best non-GT candidate by avg_logprob

then compute the same style of margins:
*   avg_margin = gold_avg_logprob - bias_avg_logprob
*   seq_margin = gold_seq_logprob - bias_seq_logprob
*   char_margin = gold_char_logprob - bias_char_logprob
*   pmi_margin = gold_pmi - bias_pmi

Outputs:
- `original_sequence_model_outputs.csv`
- `original_sequence_candidates.csv`
- `original_sequence_results.csv`
- `original_sequence_skipped.csv`
- `original_sequence_results_with_pmi.csv`


In [1]:
# Check GPU.
!nvidia-smi

Thu May 14 21:15:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install dependencies.
!pip install -q transformers accelerate pillow datasets

In [3]:
import gc
import re
import json
import math
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm
from PIL import Image

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

### Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# HuggingFace login
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Configuration

In [6]:
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
DATASET_NAME = "anvo25/vlms-are-biased"
SPLIT_NAME = "original"

OUTPUT_DIR = Path("/content/gdrive/MyDrive/original_sequence_outputs/qwen7b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 2
MIN_CANDIDATE = 0
EXCLUDE_OPTICAL_ILLUSIONS = True

# Use 20 for smoke test; None for full run.
MAX_SAMPLES = None

# Sequence pipeline is numeric-only.
RUN_ID_ROWS = False

# Conservative settings for 7B on free Colab/T4.
BATCH_GEN_NUMERIC = 1
BATCH_CANDIDATES = 1

MAX_NEW_NUMERIC = 3

# Used only for free-generation diagnostic output.
# Teacher-forced candidate scoring uses the raw dataset prompt.
GENERATION_NUMERIC_SUFFIX = "\nAnswer with exactly one integer. No words."

SAVE_EVERY = 25

print("MODEL:", MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CUDA:", torch.cuda.is_available())

MODEL: Qwen/Qwen2.5-VL-7B-Instruct
OUTPUT_DIR: /content/gdrive/MyDrive/original_sequence_outputs/qwen7b
CUDA: True


## Load Model

In [7]:
# Load model and processor
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32
print("CUDA available:", torch.cuda.is_available())

gc.collect()
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

CUDA available: True
GPU: Tesla T4


In [8]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if device == "cuda" else None,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Required for batched generation with decoder-only models.
processor.tokenizer.padding_side = "left"
tokenizer = processor.tokenizer

# Make sure a pad token exists.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

print("Loaded model:", MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded model: Qwen/Qwen2.5-VL-7B-Instruct


## Load original split from HuggingFace

In [9]:
ds = load_dataset(DATASET_NAME, split=SPLIT_NAME)
print("Original rows:", len(ds))

if EXCLUDE_OPTICAL_ILLUSIONS:
    before = len(ds)
    ds = ds.filter(
        lambda x: str(x.get("topic", "")).strip().lower() != "optical illusion"
    )
    print("Removed Optical Illusion rows:", before - len(ds))

if MAX_SAMPLES is not None:
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))
    print("Debug subset rows:", len(ds))

if "idx" in ds.column_names:
    ds = ds.remove_columns(["idx"])

ds = ds.add_column("idx", list(range(len(ds))))

assert ds["idx"][0] == 0
assert ds["idx"][-1] == len(ds) - 1
assert len(set(ds["idx"])) == len(ds)

print("Final rows:", len(ds))
print("Topics:", Counter(ds["topic"]))

README.md: 0.00B [00:00, ?B/s]

data/main-00000-of-00002.parquet:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

data/main-00001-of-00002.parquet:   0%|          | 0.00/375M [00:00<?, ?B/s]

data/identification-00000-of-00001.parqu(…):   0%|          | 0.00/391M [00:00<?, ?B/s]

data/withtitle-00000-of-00001.parquet:   0%|          | 0.00/451M [00:00<?, ?B/s]

...

Original rows: 458


Filter:   0%|          | 0/458 [00:00<?, ? examples/s]

Removed Optical Illusion rows: 132


Flattening the indices:   0%|          | 0/326 [00:00<?, ? examples/s]

Final rows: 326
Topics: Counter({'Animals': 182, 'Logos': 94, 'Flags': 38, 'Game Boards': 8, 'Chess Pieces': 4})


## Helper Functions

In [10]:
BRACE_RE = re.compile(r"\{([^{}]+)\}")
INT_RE = re.compile(r"-?\d+")


def safe_text(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()

In [11]:
def first_int(x):
    text = safe_text(x)
    m_brace = BRACE_RE.search(text)
    text = m_brace.group(1).strip() if m_brace else text
    m_int = INT_RE.search(text)
    return int(m_int.group(0)) if m_int else None


def normalized(x):
    text = safe_text(x)
    m = BRACE_RE.search(text)
    text = m.group(1).strip() if m else text
    return text.lower().strip().rstrip(".")

In [12]:
def is_numeric_row(row):
    qtype = safe_text(row.get("type_of_question", "")).lower()
    return ("count" in qtype or "illusion" in qtype) and first_int(row.get("ground_truth")) is not None


def make_base_row(row, prompt_used, answer, error=""):
    gt_raw = row.get("ground_truth", "")
    gt = first_int(gt_raw)
    pred = first_int(answer)

    return {
        "idx": int(row["idx"]),
        "ID": row.get("ID"),
        "image_path": row.get("image_path"),
        "topic": row.get("topic"),
        "sub_topic": row.get("sub_topic"),
        "question_type": safe_text(row.get("type_of_question", "")),
        "type_of_question": safe_text(row.get("type_of_question", "")),
        "prompt": safe_text(row.get("prompt", "")),
        "prompt_used_for_generation": prompt_used,
        "ground_truth": str(gt) if gt is not None else safe_text(gt_raw),
        "ground_truth_raw": gt_raw,
        "ground_truth_num": gt,
        "model_answer": answer,
        "pred_norm": normalized(answer),
        "gt_norm": normalized(gt_raw),
        "pred_num": pred,
        "correct_exact": normalized(answer) == normalized(gt_raw),
        "correct_numeric": int(pred) == int(gt) if pred is not None and gt is not None else False,
        "generation_error": error,
    }

In [13]:
def chat_template(image, prompt):
    return processor.apply_chat_template(
        [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }],
        tokenize=False,
        add_generation_prompt=True,
    )


def entropy_from_logprobs(log_probs):
    probs = log_probs.exp()
    return float(-(probs * log_probs).sum().item())

### Save Outputs functions

In [14]:
def sort_key(row):
    return 10**12 if row.get("idx") is None else int(row["idx"])


def read_csv_records(path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        try:
            return pd.read_csv(path).to_dict("records")
        except pd.errors.EmptyDataError:
            return []
    return []

In [15]:
numeric_output_rows = read_csv_records(
    OUTPUT_DIR / "original_sequence_numeric_outputs.csv"
)
candidate_rows = read_csv_records(
    OUTPUT_DIR / "original_sequence_candidates.csv"
)
result_rows = read_csv_records(
    OUTPUT_DIR / "original_sequence_results.csv"
)
skipped_rows = read_csv_records(
    OUTPUT_DIR / "original_sequence_skipped.csv"
)

print("Loaded checkpoint:")
print("numeric_output_rows:", len(numeric_output_rows))
print("candidate_rows:", len(candidate_rows))
print("result_rows:", len(result_rows))
print("skipped_rows:", len(skipped_rows))

Loaded checkpoint:
numeric_output_rows: 0
candidate_rows: 0
result_rows: 0
skipped_rows: 0


In [16]:
def save_outputs():
    pd.DataFrame(sorted(numeric_output_rows, key=sort_key)).to_csv(
        OUTPUT_DIR / "original_sequence_numeric_outputs.csv",
        index=False,
    )

    pd.DataFrame(
        sorted(candidate_rows, key=lambda r: (sort_key(r), int(r.get("candidate", -1))))
    ).to_csv(
        OUTPUT_DIR / "original_sequence_candidates.csv",
        index=False,
    )

    pd.DataFrame(sorted(result_rows, key=sort_key)).to_csv(
        OUTPUT_DIR / "original_sequence_results.csv",
        index=False,
    )

    pd.DataFrame(sorted(skipped_rows, key=sort_key)).to_csv(
        OUTPUT_DIR / "original_sequence_skipped.csv",
        index=False,
    )

## Bach Free Generation

In [17]:
def generate_batch(rows, max_new_tokens, need_scores=True):
    prepared = []

    for row in rows:
        row = dict(row)

        prompt_raw = safe_text(row.get("prompt", ""))
        prompt_used = prompt_raw + GENERATION_NUMERIC_SUFFIX

        try:
            image = row["image"].convert("RGB")
            text = chat_template(image, prompt_used)
            error = ""
        except Exception as e:
            image, text, error = None, None, repr(e)

        prepared.append({
            "row": row,
            "prompt_used": prompt_used,
            "image": image,
            "text": text,
            "error": error,
        })

    results = [None] * len(prepared)
    valid = [i for i, p in enumerate(prepared) if not p["error"]]

    for i, p in enumerate(prepared):
        if p["error"]:
            results[i] = {
                "row": p["row"],
                "prompt_used": p["prompt_used"],
                "answer": "",
                "gen_ids": None,
                "gen_logprobs": None,
                "error": p["error"],
            }

    if not valid:
        return results

    inputs = processor(
        text=[prepared[i]["text"] for i in valid],
        images=[prepared[i]["image"] for i in valid],
        padding=True,
        return_tensors="pt",
    ).to(device)

    try:
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=need_scores,
                pad_token_id=tokenizer.pad_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]

        gen_logprobs = None
        if need_scores and len(output.scores) > 0:
            gen_logprobs = torch.log_softmax(
                torch.stack(output.scores, dim=0).float(),
                dim=-1,
            )

        for batch_i, original_i in enumerate(valid):
            gen_ids = output.sequences[batch_i, prompt_len:].detach().cpu()
            answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            results[original_i] = {
                "row": prepared[original_i]["row"],
                "prompt_used": prepared[original_i]["prompt_used"],
                "answer": answer,
                "gen_ids": gen_ids,
                "gen_logprobs": gen_logprobs[:, batch_i, :].detach().cpu() if need_scores else None,
                "error": "",
            }

        del output
        if gen_logprobs is not None:
            del gen_logprobs

    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        for original_i in valid:
            results[original_i] = {
                "row": prepared[original_i]["row"],
                "prompt_used": prepared[original_i]["prompt_used"],
                "answer": "",
                "gen_ids": None,
                "gen_logprobs": None,
                "error": "cuda_oom_generation",
            }

    finally:
        del inputs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

## Score Candidates

In [18]:
def score_candidate_sequence(image, prompt, answer):
    """
    Main-pipeline-style teacher-forced sequence scoring.

    Scores:
      full_text = chat_template(image, prompt) + answer

    Only answer tokens contribute to:
      seq_logprob, avg_logprob, char_logprob, entropy_step1, entropy_mean

    Memory-safe:
      does not convert the full logits tensor to fp32.
      converts one vocab vector at a time.
    """
    prompt_text = chat_template(image, prompt)
    answer_text = str(answer)
    answer_ids_expected = tokenizer.encode(answer_text, add_special_tokens=False)
    full_text = prompt_text + answer_text

    prompt_inputs = processor(
        text=[prompt_text],
        images=[image],
        return_tensors="pt",
    )
    prompt_len = prompt_inputs["input_ids"].shape[1]

    inputs = None
    output = None

    try:
        inputs = processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            output = model(**inputs)

        input_ids = inputs["input_ids"]
        ids = input_ids[0].detach().cpu().tolist()

        span_start = prompt_len
        span_end = prompt_len + len(answer_ids_expected)
        span_mode = "exact"

        # Defensive fallback if token span is shifted.
        if ids[span_start:span_end] != answer_ids_expected:
            span_start = None
            search_from = max(0, prompt_len - 5)

            for pos in range(search_from, len(ids) - len(answer_ids_expected) + 1):
                if ids[pos:pos + len(answer_ids_expected)] == answer_ids_expected:
                    span_start = pos
                    span_end = pos + len(answer_ids_expected)
                    span_mode = "fallback"
                    break

            if span_start is None:
                return {
                    "status": "failed",
                    "span_mode": "failed",
                    "error": "candidate_span_not_found",
                    "candidate_token_ids": json.dumps(answer_ids_expected),
                }

        token_lps = []
        token_entropies = []

        for pos in range(span_start, span_end):
            if pos == 0:
                return {
                    "status": "failed",
                    "span_mode": "failed",
                    "error": "span_starts_at_zero",
                    "candidate_token_ids": json.dumps(answer_ids_expected),
                }

            tok_id = int(input_ids[0, pos].item())

            pred_logits = output.logits[0, pos - 1, :].float()
            pred_log_probs = torch.log_softmax(pred_logits, dim=-1)

            token_lps.append(float(pred_log_probs[tok_id].item()))

            probs = pred_log_probs.exp()
            token_entropies.append(float(-(probs * pred_log_probs).sum().item()))

            del pred_logits, pred_log_probs, probs

        seq_lp = float(sum(token_lps))
        n_tokens = len(token_lps)

        return {
            "status": "ok",
            "span_mode": span_mode,
            "seq_logprob": seq_lp,
            "avg_logprob": seq_lp / max(1, n_tokens),
            "char_logprob": seq_lp / max(1, len(answer_text)),
            "entropy_step1": token_entropies[0] if token_entropies else None,
            "entropy_mean": float(np.mean(token_entropies)) if token_entropies else None,
            "n_tokens": n_tokens,
            "candidate_token_ids": json.dumps(answer_ids_expected),
            "token_logprobs": json.dumps(token_lps),
        }

    except torch.cuda.OutOfMemoryError:
        return {
            "status": "failed",
            "span_mode": "failed",
            "error": "cuda_oom_sequence_scoring",
            "candidate_token_ids": json.dumps(answer_ids_expected),
        }

    finally:
        try:
            del inputs, output
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [19]:
def score_candidate_window(image, prompt, candidates):
    scored = []

    for candidate in candidates:
        score = score_candidate_sequence(
            image=image,
            prompt=prompt,
            answer=str(candidate),
        )

        scored.append({
            "candidate": int(candidate),
            "candidate_str": str(candidate),
            **score,
        })

    return scored

### Numerical pipeline

In [20]:
def analyze_numeric_sequence(result):
    row = result["row"]
    gt = first_int(row.get("ground_truth", ""))

    if gt is None:
        return None, [], {"skip_reason": "non_numeric_ground_truth"}

    if result["error"]:
        return None, [], {"skip_reason": "generation_error"}

    try:
        image = row["image"].convert("RGB")
    except Exception as e:
        return None, [], {"skip_reason": "image_load_error", "error": repr(e)}

    # Main-sequence consistency:
    # Candidate scoring uses raw dataset prompt, not generation suffix.
    scoring_prompt = safe_text(row.get("prompt", ""))

    candidates = list(range(max(MIN_CANDIDATE, gt - WINDOW), gt + WINDOW + 1))
    scored = score_candidate_window(image, scoring_prompt, candidates)
    ok = [x for x in scored if x.get("status") == "ok"]

    if not ok:
        return None, scored, {
            "skip_reason": "all_candidate_spans_failed",
            "candidate_set": json.dumps(candidates),
        }

    gold_rows = [x for x in ok if int(x["candidate"]) == int(gt)]
    non_gold = [x for x in ok if int(x["candidate"]) != int(gt)]

    if not gold_rows:
        return None, scored, {
            "skip_reason": "gold_span_failed",
            "candidate_set": json.dumps(candidates),
        }

    if not non_gold:
        return None, scored, {
            "skip_reason": "no_valid_non_gold_candidate",
            "candidate_set": json.dumps(candidates),
        }

    gold = gold_rows[0]
    proxy = max(non_gold, key=lambda x: float(x["avg_logprob"]))

    ranked = sorted(ok, key=lambda x: float(x["avg_logprob"]), reverse=True)
    gt_rank = next(
        i + 1 for i, x in enumerate(ranked)
        if int(x["candidate"]) == int(gt)
    )

    # Free-generation answer step and entropy diagnostics.
    answer_step = None
    detected_digit = None
    detected_token_text = None
    entropy_step1 = None
    entropy_mean = None

    if result["gen_ids"] is not None:
        for i, token_id in enumerate(result["gen_ids"]):
            token_text = tokenizer.decode([int(token_id)], skip_special_tokens=True)
            detected = first_int(token_text)
            if detected is not None:
                answer_step = i
                detected_digit = detected
                detected_token_text = token_text
                break

    if (
        answer_step is not None
        and result["gen_logprobs"] is not None
        and answer_step < result["gen_logprobs"].shape[0]
    ):
        entropy_step1 = entropy_from_logprobs(result["gen_logprobs"][answer_step].float())
        entropy_mean = float(np.mean([
            entropy_from_logprobs(result["gen_logprobs"][t].float())
            for t in range(answer_step + 1)
        ]))

    base = make_base_row(row, result["prompt_used"], result["answer"], result["error"])

    seq_margin = float(gold["seq_logprob"] - proxy["seq_logprob"])
    avg_margin = float(gold["avg_logprob"] - proxy["avg_logprob"])
    char_margin = float(gold["char_logprob"] - proxy["char_logprob"])

    candidate_scores = torch.tensor([x["avg_logprob"] for x in ok], dtype=torch.float32)
    candidate_probs = torch.softmax(candidate_scores, dim=0)
    candidate_log_probs = torch.log_softmax(candidate_scores, dim=0)
    entropy_candidates = float(-(candidate_probs * candidate_log_probs).sum().item())

    same_length = int(gold["n_tokens"]) == int(proxy["n_tokens"])

    result_row = {
        **base,

        "expected_bias": str(proxy["candidate"]),
        "proxy_bias_value": int(proxy["candidate"]),
        "proxy_bias_definition": f"best non-GT candidate from GT±{WINDOW} by avg_logprob",

        "candidate_policy": f"GT±{WINDOW}",
        "candidate_set": json.dumps(candidates),
        "num_candidates_attempted": len(candidates),
        "num_candidates_scored": len(ok),

        "answer_step": answer_step,
        "detected_digit": detected_digit,
        "detected_token_text": detected_token_text,

        "gold_seq_logprob": gold["seq_logprob"],
        "gold_avg_logprob": gold["avg_logprob"],
        "gold_char_logprob": gold["char_logprob"],
        "gold_entropy_step1": gold["entropy_step1"],
        "gold_entropy_mean": gold["entropy_mean"],
        "gold_n_tokens": gold["n_tokens"],
        "gold_span_mode": gold["span_mode"],

        "bias_seq_logprob": proxy["seq_logprob"],
        "bias_avg_logprob": proxy["avg_logprob"],
        "bias_char_logprob": proxy["char_logprob"],
        "bias_entropy_step1": proxy["entropy_step1"],
        "bias_entropy_mean": proxy["entropy_mean"],
        "bias_n_tokens": proxy["n_tokens"],
        "bias_span_mode": proxy["span_mode"],

        "seq_margin": seq_margin,
        "avg_margin": avg_margin,
        "char_margin": char_margin,

        "gt_rank_seq": gt_rank,
        "prefers_gold": avg_margin > 0,
        "prefers_gt_seq": avg_margin > 0,
        "entropy_candidates": entropy_candidates,

        "entropy_step1": entropy_step1,
        "entropy_mean": entropy_mean,

        "generated_matches_proxy": (
            detected_digit is not None
            and int(detected_digit) == int(proxy["candidate"])
        ),
        "full_hallucination": (
            detected_digit is not None
            and int(detected_digit) != int(gt)
            and int(detected_digit) != int(proxy["candidate"])
        ),

        "same_length": same_length,
        "length_matched": same_length,
        "relative_change": abs(float(gt) - float(proxy["candidate"])) / max(1.0, abs(float(proxy["candidate"]))),
        "gt_minus_bias": float(gt) - float(proxy["candidate"]),
        "direction": (
            "GT>proxy" if float(gt) > float(proxy["candidate"])
            else "GT<proxy" if float(gt) < float(proxy["candidate"])
            else "GT=proxy"
        ),
    }

    return result_row, scored, None

## PMI

In [21]:
def score_unconditional(answer):
    prompt = "Answer:"
    answer = str(answer)
    full_text = prompt + answer

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(device)

    full_ids = tokenizer(
        full_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(device)

    prompt_len = prompt_ids.shape[1]
    output = None

    try:
        with torch.inference_mode():
            output = model(input_ids=full_ids)

        answer_ids = full_ids[0, prompt_len:]
        token_lps = []

        for i, token_id in enumerate(answer_ids):
            pos = prompt_len - 1 + i
            pred_logits = output.logits[0, pos, :].float()
            pred_log_probs = torch.log_softmax(pred_logits, dim=-1)
            token_lps.append(float(pred_log_probs[token_id].item()))
            del pred_logits, pred_log_probs

        seq_lp = float(sum(token_lps))
        n_tokens = len(token_lps)

        return {
            "seq_logprob": seq_lp,
            "avg_logprob": seq_lp / max(1, n_tokens),
            "char_logprob": seq_lp / max(1, len(answer)),
            "n_tokens": n_tokens,
        }

    finally:
        try:
            del prompt_ids, full_ids, output
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [22]:
def add_pmi_columns(rows):
    df = pd.DataFrame(rows)

    if len(df) == 0:
        return rows

    unique_values = sorted(
        set(df["ground_truth"].astype(str))
        | set(df["expected_bias"].astype(str))
    )

    print(f"Running unconditional scoring for {len(unique_values)} unique values:")
    print(unique_values)

    uncond = {}

    for value in tqdm(unique_values, desc="PMI"):
        uncond[str(value)] = score_unconditional(str(value))

    df["gold_uncond_avg_lp"] = df["ground_truth"].astype(str).map(
        {k: v["avg_logprob"] for k, v in uncond.items()}
    )
    df["bias_uncond_avg_lp"] = df["expected_bias"].astype(str).map(
        {k: v["avg_logprob"] for k, v in uncond.items()}
    )

    df["gold_uncond_char_lp"] = df["ground_truth"].astype(str).map(
        {k: v["char_logprob"] for k, v in uncond.items()}
    )
    df["bias_uncond_char_lp"] = df["expected_bias"].astype(str).map(
        {k: v["char_logprob"] for k, v in uncond.items()}
    )

    df["gold_pmi"] = df["gold_avg_logprob"] - df["gold_uncond_avg_lp"]
    df["bias_pmi"] = df["bias_avg_logprob"] - df["bias_uncond_avg_lp"]
    df["pmi_margin"] = df["gold_pmi"] - df["bias_pmi"]
    df["prefers_gold_pmi"] = df["pmi_margin"] > 0

    df["gold_char_pmi"] = df["gold_char_logprob"] - df["gold_uncond_char_lp"]
    df["bias_char_pmi"] = df["bias_char_logprob"] - df["bias_uncond_char_lp"]
    df["char_pmi_margin"] = df["gold_char_pmi"] - df["bias_char_pmi"]
    df["prefers_gold_char_pmi"] = df["char_pmi_margin"] > 0

    return df.to_dict("records")

## Run Benchmark

In [23]:
all_numeric_rows = []
ignored_id_rows = []

for row in ds:
    row = dict(row)
    if is_numeric_row(row):
        all_numeric_rows.append(row)
    else:
        ignored_id_rows.append(row)

In [24]:
done_sequence_idx = {
    int(r["idx"]) for r in result_rows
    if pd.notna(r.get("idx"))
}

done_sequence_idx |= {
    int(r["idx"]) for r in skipped_rows
    if r.get("pipeline") == "sequence" and pd.notna(r.get("idx"))
}

numeric_rows = [
    r for r in all_numeric_rows
    if int(r["idx"]) not in done_sequence_idx
]

In [25]:
print("Total numeric rows:", len(all_numeric_rows))
print("Pending numeric rows:", len(numeric_rows))
print("Ignored ID/text rows:", len(ignored_id_rows))

processed = len(done_sequence_idx)
next_save = ((processed // SAVE_EVERY) + 1) * SAVE_EVERY

Total numeric rows: 163
Pending numeric rows: 163
Ignored ID/text rows: 163


In [26]:
for start in tqdm(
    range(0, len(numeric_rows), BATCH_GEN_NUMERIC),
    desc="Numeric sequence",
):
    batch = numeric_rows[start:start + BATCH_GEN_NUMERIC]
    results = generate_batch(batch, MAX_NEW_NUMERIC, need_scores=True)

    for result in results:
        row = result["row"]
        base = make_base_row(row, result["prompt_used"], result["answer"], result["error"])
        numeric_output_rows.append(base)

        if result["error"]:
            skipped_rows.append({
                **base,
                "pipeline": "sequence",
                "skip_reason": "generation_error",
            })
        else:
            sequence_row, scored_candidates, skip_info = analyze_numeric_sequence(result)

            for candidate in scored_candidates:
                candidate_rows.append({
                    "idx": int(row["idx"]),
                    "ID": row.get("ID"),
                    "topic": row.get("topic"),
                    "sub_topic": row.get("sub_topic"),
                    "ground_truth_num": first_int(row.get("ground_truth")),
                    **candidate,
                })

            if sequence_row is not None:
                result_rows.append(sequence_row)
            else:
                skipped_rows.append({
                    **base,
                    "pipeline": "sequence",
                    **(skip_info or {}),
                })

        processed += 1

        if processed >= next_save:
            save_outputs()
            print(f"Saved at {processed} numeric rows.")
            next_save += SAVE_EVERY

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Numeric sequence:   0%|          | 0/163 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved at 25 numeric rows.
Saved at 50 numeric rows.
Saved at 75 numeric rows.
Saved at 100 numeric rows.
Saved at 125 numeric rows.
Saved at 150 numeric rows.


## Finalize

In [27]:
if len(numeric_output_rows):
    numeric_output_rows = pd.DataFrame(numeric_output_rows).drop_duplicates(
        "idx",
        keep="last",
    ).to_dict("records")

if len(result_rows):
    result_rows = pd.DataFrame(result_rows).drop_duplicates(
        "idx",
        keep="last",
    ).to_dict("records")

if len(skipped_rows):
    skipped_rows = pd.DataFrame(skipped_rows).drop_duplicates(
        ["idx", "pipeline"],
        keep="last",
    ).to_dict("records")

if len(candidate_rows):
    candidate_rows = pd.DataFrame(candidate_rows).drop_duplicates(
        ["idx", "candidate"],
        keep="last",
    ).to_dict("records")

In [28]:
# Add/re-add PMI after all sequence result rows are complete.
result_rows = add_pmi_columns(result_rows)

numeric_output_rows = sorted(numeric_output_rows, key=sort_key)
candidate_rows = sorted(candidate_rows, key=lambda r: (sort_key(r), int(r.get("candidate", -1))))
result_rows = sorted(result_rows, key=sort_key)
skipped_rows = sorted(skipped_rows, key=sort_key)

numeric_idx = {int(r["idx"]) for r in all_numeric_rows}
output_idx = {int(r["idx"]) for r in numeric_output_rows}
result_idx = {int(r["idx"]) for r in result_rows}
skip_idx = {
    int(r["idx"]) for r in skipped_rows
    if r.get("pipeline") == "sequence"
}

Running unconditional scoring for 20 unique values:
['1', '10', '11', '12', '13', '14', '18', '19', '2', '3', '30', '32', '4', '48', '5', '50', '6', '7', '8', '9']


PMI:   0%|          | 0/20 [00:00<?, ?it/s]

In [29]:
assert output_idx | skip_idx == numeric_idx, "Some numeric rows have no generation output or skip record."
assert result_idx | skip_idx == numeric_idx, "Some numeric rows are neither scored nor skipped."

save_outputs()

In [30]:
# Save PMI-enriched final results.
pd.DataFrame(result_rows).to_csv(
    OUTPUT_DIR / "original_sequence_results_with_pmi.csv",
    index=False,
)

In [31]:
print("\nDone.")
print("Numeric rows:", len(all_numeric_rows))
print("Ignored ID/text rows:", len(ignored_id_rows))
print("Numeric output rows:", len(numeric_output_rows))
print("Candidate rows:", len(candidate_rows))
print("Sequence result rows:", len(result_rows))
print("Sequence skipped rows:", len(skip_idx))

print("\nSaved:")
print(OUTPUT_DIR / "original_sequence_numeric_outputs.csv")
print(OUTPUT_DIR / "original_sequence_candidates.csv")
print(OUTPUT_DIR / "original_sequence_results.csv")
print(OUTPUT_DIR / "original_sequence_results_with_pmi.csv")
print(OUTPUT_DIR / "original_sequence_skipped.csv")


Done.
Numeric rows: 163
Ignored ID/text rows: 163
Numeric output rows: 163
Candidate rows: 814
Sequence result rows: 163
Sequence skipped rows: 0

Saved:
/content/gdrive/MyDrive/original_sequence_outputs/qwen7b/original_sequence_numeric_outputs.csv
/content/gdrive/MyDrive/original_sequence_outputs/qwen7b/original_sequence_candidates.csv
/content/gdrive/MyDrive/original_sequence_outputs/qwen7b/original_sequence_results.csv
/content/gdrive/MyDrive/original_sequence_outputs/qwen7b/original_sequence_results_with_pmi.csv
/content/gdrive/MyDrive/original_sequence_outputs/qwen7b/original_sequence_skipped.csv
